In [ ]:
PRED  (Entity, Pred)
  └── PRED-LIST  (PredList = List{Pred})
        └── ILP-CORE  (BK, POS, NEG : PredList)
              └── METAVAR  (SecondOrderVar, FirstOrderVar)
                    └── METALITERAL-BASE  ([P:A,B])
                          └── METALITERAL  (MetaBody)
                                └── METARULE-BASE  (Metarule)
                                      └── METARULE  (MetaruleList, METARULES)
                                            └── GRANDPARENT-ILP-NEW2  (bài toán cụ thể)
                                                  └── ILP-EXTRACT  (engine đọc cấu trúc)

## MODULE TRÍCH XUẤT SIÊU MẪU (ILP-EXTRACT)
1. Tiện ích dành cho TermList (Duyệt đối số)
Trong Meta-level, khi bạn có một vị từ như mother(ann, amy), thì (ann, amy) là một TermList. Nhóm hàm này giúp bạn thao tác với danh sách đó giống như cách chúng ta xử lý mảng hoặc danh sách liên kết.

headTL & tailTL: Đây là bộ đôi "Đầu và Đuôi".

headTL lấy ra phần tử đầu tiên.

tailTL vứt phần tử đầu đi, lấy toàn bộ phần còn lại.

lengthTL: Đếm xem danh sách có bao nhiêu phần tử. Trong ILP, hàm này cực kỳ quan trọng để kiểm tra Arity (số lượng đối số) của một quan hệ.

nthTL: Lấy phần tử ở vị trí thứ N. Ví dụ: trong likes(bob, pizza), phần tử thứ 0 là bob, phần tử thứ 1 là pizza.

appendTL: Nối hai danh sách lại với nhau.

Tóm lại: Phần 1 xây dựng các công cụ cơ bản để bạn "đi bộ" xuyên qua danh sách các biến hoặc thực thể.

2. Trích xuất cơ bản (Functor & Arguments)
Phần này dùng để "mổ xẻ" một cấu trúc f(x, y).

functorOf: Trả về cái Tên của hàm/vị từ (gọi là Qid).

Ví dụ: functorOf(mother(ann, amy)) sẽ trả về 'mother.

argsOf: Trả về cái Ruột bên trong ngoặc.

Ví dụ: argsOf(mother(ann, amy)) sẽ trả về danh sách ('ann, 'amy).

flattenGeneric (Cực kỳ quan trọng):

Vấn đề: Maude xử lý các toán tử như dấu phẩy , hoặc dấu cách (trống) theo cấu trúc cây nhị phân (ví dụ: A, B, C sẽ bị hiểu là (A, (B, C))).

Giải pháp: Hàm này sẽ "ủi phẳng" cái cây đó. Nó đi sâu vào các nhánh, nếu gặp đúng ký hiệu (như dấu ,), nó sẽ lôi các phần tử ra và xếp chúng thành một hàng ngang (TermList) đơn giản. Nếu không có hàm này, bạn sẽ rất dễ bị sót dữ liệu khi duyệt luật.

3. Trích xuất Metarule (Vị từ, Head, Body)
Đây là nơi chúng ta áp dụng các công cụ trên vào cấu trúc cụ thể của một Metarule (Siêu luật). Một Metarule thường có dạng: metarule(Biến_vị_từ, Head, Body).

mrPredVars: Tìm các biến vị từ như P,Q,R.

Nó thực hiện một chuỗi các bước "nhảy": lấy đối số đầu tiên của Metarule, sau đó lại lấy đối số của đối số đó... rồi cuối cùng dùng flattenGeneric với dấu |_| để lấy ra danh sách 'P, 'Q, 'R.

mrHead & mrBody: Tách luật thành hai phần:

mrHead: Lấy phần hệ quả (ví dụ: [P : A, B]).

mrBody: Lấy phần điều kiện (ví dụ: [Q : A, C] [R : C, B]).

mrBodyLiterals:

Phần thân (Body) của luật thường chứa nhiều Literal viết cách nhau bằng dấu cách (toán tử __).

Hàm này dùng flattenGeneric với ký hiệu '__ để tách chúng ra thành một danh sách các Literal riêng lẻ. Nhờ đó, máy tính có thể kiểm tra từng điều kiện một.

4. Trích xuất Literal & Fact (Đối số và Vị từ)
Phần này giúp máy tính phân biệt được đâu là "Cái khung" (Meta-Literal) và đâu là "Thực tế" (Fact).

mlPredVar: Lấy biến vị từ của một Literal.

Ví dụ: Với Literal [P : A, B], hàm này lấy ra thằng P.

mlArgVarList: Lấy danh sách các biến đối số.

Nó dùng flattenGeneric với dấu phẩy (_,_) để đảm bảo dù bạn viết A, BhayA, B, C, D`, nó đều trả về một danh sách phẳng để dễ xử lý.

mlArity: Đếm số lượng đối số trong Literal.

factPred (CỰC KỲ QUAN TRỌNG):

Vấn đề: Trong Maude Meta-level, biến vị từ thường có Sort là Pred (ví dụ: 'P:Pred). Nhưng tên hàm trong Fact chỉ là một chuỗi (ví dụ: 'mother). Hai cái này không "khớp" nhau về kiểu dữ liệu.

Giải pháp: Dòng qid(string(F:Qid) + ".Pred") sẽ "phù phép" biến tên 'mother thành 'mother.Pred. Lúc này, máy tính mới hiểu: "À, hằng số mother này có thể lắp vừa vào vị trí của biến P".

arityMatch: Kiểm tra "số ghế" (Arity của Literal) có bằng "số người" (Arity của Fact) không. Nếu Literal cần 2 biến mà Fact có 3 đối số, nó sẽ loại ngay từ vòng gửi xe.

5. Substitution (Bộ gán biến & Lan truyền lỗi)
Đây chính là "Bộ nhớ tạm" của thuật toán ILP. Nó lưu giữ câu trả lời cho câu hỏi: "Biến A hiện giờ đang đại diện cho ai?".

A. Định nghĩa cấu trúc
Binding: Một cặp gán đơn lẻ, ví dụ: bind(A, ann) (nghĩa là A là ann).

Subst: Một danh sách các Binding.

nilSubst: Bộ nhớ trống (trạng thái bắt đầu).

failSubst: Tín hiệu báo động đỏ. Khi phát hiện mâu thuẫn logic, toàn bộ bộ nhớ sẽ biến thành failSubst.

B. Logic "Lan truyền lỗi" (Fail-fast)
Đoạn mã
eq failSubst :: S:Subst = failSubst .
Dòng này cực kỳ thông minh. Nó có nghĩa là: "Nếu trong quá trình chứng minh một luật có 3 bước, mà bước 1 đã Thất bại (failSubst), thì không cần làm bước 2 và 3 nữa, kết quả cuối cùng chắc chắn là Thất bại".

C. Hàm extendSubst (Trái tim của sự suy diễn)
Đây là nơi thực hiện phép gán biến mới vào bộ nhớ. Nó chạy theo logic sau:

Nếu đã hỏng (failSubst): Trả về hỏng luôn.

Nếu biến T1 đã có giá trị trong bộ nhớ (isBound):

Kiểm tra xem giá trị cũ có giống giá trị mới (T2) không.

Nếu giống: Giữ nguyên bộ nhớ (vì không có gì mới).

Nếu khác (Xung đột): Trả về failSubst. Ví dụ: A đang là ann mà lại bảo A là bob thì là sai.

Nếu biến T1 chưa có giá trị:

Ghi tên T1 và giá trị T2 vào bộ nhớ: bind(T1, T2) :: S.

Tổng kết quy trình chạy của 2 phần này:
Giả sử bạn đang thử khớp luật [P : A, B] với Fact mother(ann, amy):

Bước 1 (Phần 4): factPred biến mother thành mother.Pred. mlPredVar lấy ra P.

Bước 2 (Phần 5): Gọi extendSubst(P, mother.Pred, nilSubst). Bộ nhớ thành {P:mother}.

Bước 3 (Phần 4): mlArgVarList lấy (A, B). argsOf lấy (ann, amy).

Bước 4 (Phần 5): Gọi extendSubst lần lượt cho A với ann và B với amy.

Kết quả: Bạn nhận được bộ nhớ {P:mother, A:ann, B:amy}.